# 📚 Создание RAG-индекса медиатеки

Этот блокнот создаёт векторный индекс для семантического поиска фильмов и сериалов.

**Что делает:**
- Подключается к Google Gemini API
- Загружает данные из SQLite базы медиатеки
- Создаёт эмбеддинги через Gemini Embedding API
- Сохраняет индекс в SQLite для локального использования

**Требования:**
- GEMINI_API_KEY в переменных окружения
- Файл media.db с медиатекой

In [ ]:
# @title 📦 Установка зависимостей
!pip install -q google-genai numpy

In [ ]:
# @title 🔑 Настройка API ключа
import os
from google.colab import userdata

# Введите ваш Gemini API Key ниже или используйте Secret
api_key = userdata.get('GEMINI_API_KEY') or ''

if not api_key:
    api_key = input('Введите GEMINI_API_KEY: ').strip()

os.environ['GEMINI_API_KEY'] = api_key
print('✅ API ключ настроен')

In [ ]:
# @title 📂 Загрузка базы данных медиатеки

from google.colab import files
import sqlite3
from pathlib import Path
import json

# Загрузка media.db с локальной машины
print("📤 Загрузите файл media.db с вашей медиатекой:")
uploaded = files.upload()

db_file = None
for fname in uploaded.keys():
    if fname.endswith('.db'):
        db_file = Path(fname)
        print(f'✅ Загружен: {fname}')
        break

if not db_file:
    raise FileNotFoundError('Файл media.db не найден!')

## 🧠 Создание RAG-индекса

In [ ]:
# @title ⚙️ Инициализация RAG

import json
import sqlite3
from pathlib import Path
from typing import List, Dict

_EMBED_MODEL = 'models/text-embedding-004'
_EMBED_DIM = 768

# Создание клиента Gemini напрямую через requests
import requests

def get_embeddings(texts: List[str], api_key: str) -> List[List[float]]:
    """Получение эмбеддингов через REST API Gemini."""
    if not texts:
        return []
    
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{_EMBED_MODEL}:embedContent?key={api_key}'
    results = []
    
    for text in texts:
        payload = {
            'model': f'models/{_EMBED_MODEL}',
            'content': {'parts': [{'text': text}]},
            'embedContentConfig': {'taskType': 'RETRIEVAL_DOCUMENT'}
        }
        
        try:
            response = requests.post(url, json=payload, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if 'embedding' in data:
                    results.append(data['embedding']['values'])
                else:
                    print(f"⚠️ Нет embedding в ответе для: {text[:50]}...")
                    results.append([0.0] * _EMBED_DIM)
            else:
                print(f"❌ API ошибка {response.status_code}: {response.text[:200]}")
                results.append([0.0] * _EMBED_DIM)
        except Exception as e:
            print(f"❌ Ошибка запроса: {e}")
            results.append([0.0] * _EMBED_DIM)
    
    return results

class ColabRAG:
    """RAG-индекс для Google Colab."""

    def __init__(self, api_key: str, db_path: Path) -> None:
        self.db_path = db_path
        self.api_key = api_key
        db_path.parent.mkdir(parents=True, exist_ok=True)
        self._init_db()

    def _init_db(self) -> None:
        """Создание таблицы для эмбеддингов."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS rag_index (
                    id TEXT PRIMARY KEY,
                    text TEXT NOT NULL,
                    meta TEXT NOT NULL DEFAULT '{}',
                    embedding BLOB NOT NULL
                )
            """)

    def _record_to_text(self, record: dict) -> str:
        """Сериализация записи в текст."""
        parts = [
            (record.get('title') or ''),
            (record.get('title_ru') or ''),
            (record.get('title_orig') or ''),
            f"Год: {record.get('year') or ''}",
            f"Тип: {record.get('type') or ''}",
            f"Категория: {record.get('main_category') or ''}",
            f"Страна: {record.get('country') or ''}",
            f"Жанры: {', '.join((record.get('genres') or [])[:5])}",
            f"Режиссёры: {', '.join((record.get('directors') or [])[:3])}",
            f"В ролях: {', '.join((record.get('cast') or [])[:10])}",
            ((record.get('plot') or '') or '')[:500],
            (record.get('atmosphere') or ''),
            (record.get('why_watch') or ''),
        ]
        return ' '.join(p for p in parts if p).strip()

    def build(self, source_db: Path) -> int:
        """Построение индекса из source_db."""
        import numpy as np

        # Чтение исходной базы
        with sqlite3.connect(source_db) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute('SELECT * FROM media ORDER BY disk_name, title').fetchall()

        records = []
        for row in rows:
            data = dict(row)
            for field in ('genres', 'directors', 'cast', 'rating', 'facts', 'similar', 'review'):
                raw = data.get(field)
                if raw:
                    try:
                        data[field] = json.loads(raw)
                    except (json.JSONDecodeError, TypeError):
                        data[field] = [] if field not in ('rating', 'review') else {}
            # Проверка наличия обязательных полей
            if data.get('title'):
                records.append(data)

        print(f'📊 Загружено записей: {len(records)}')

        if not records:
            print('⚠️ Нет записей для индексации!')
            return 0

        # Очистка и перестроение
        self.clear()

        # Векторизация батчами по 50
        batch_size = 50
        total = 0

        for i in range(0, len(records), batch_size):
            batch = records[i:i + batch_size]
            texts = [self._record_to_text(r) for r in batch]
            vectors = get_embeddings(texts, self.api_key)

            with sqlite3.connect(self.db_path) as conn:
                for idx, record in enumerate(batch):
                    if idx >= len(vectors):
                        continue
                    doc_id = f"{record.get('disk_name', 'UNKNOWN')}::{record.get('type', 'unknown')}::{record.get('title', 'unknown')}"
                    meta = {
                        'title': record.get('title', ''),
                        'type': record.get('type', ''),
                        'disk_name': record.get('disk_name', ''),
                        'main_category': record.get('main_category', ''),
                        'year': record.get('year') or 0,
                    }
                    text = texts[idx]
                    conn.execute(
                        'INSERT OR REPLACE INTO rag_index (id, text, meta, embedding) VALUES (?, ?, ?, ?)',
                        (doc_id, text, json.dumps(meta, ensure_ascii=False), np.array(vectors[idx], dtype=np.float32).tobytes())
                    )
            total += len(batch)
            print(f'  Обработано: {total}/{len(records)}')

        return self.count()

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Поиск по семантической близости."""
        import numpy as np

        if self.count() == 0:
            print('⚠️ Индекс пуст! Сначала создайте индекс.')
            return []

        query_vec = np.array(get_embeddings([query], self.api_key)[0], dtype=np.float32)
        query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-10)

        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute('SELECT id, text, meta, embedding FROM rag_index').fetchall()

        results = []
        for row_id, text, meta_str, emb_bytes in rows:
            vec = np.frombuffer(emb_bytes, dtype=np.float32)
            norm = vec / (np.linalg.norm(vec) + 1e-10)
            score = float(np.dot(query_norm, norm))
            meta = json.loads(meta_str) if meta_str else {}
            results.append({'id': row_id, 'text': text[:100], 'meta': meta, 'score': round(score, 4)})

        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]

    def clear(self) -> None:
        """Очистка индекса."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('DELETE FROM rag_index')

    def count(self) -> int:
        """Количество документов."""
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute('SELECT COUNT(*) FROM rag_index').fetchone()[0]

print('✅ Класс ColabRAG готов')

In [ ]:
# @title 🚀 Запуск построения индекса

rag = ColabRAG(api_key=api_key, db_path=Path('media_rag.db'))
count = rag.build(db_file)
print(f'\n🎉 Готово! Документов в индексе: {count}')

In [ ]:
# @title 🔍 Тестовый поиск

query = input('Введите поисковый запрос: ')
if query:
    results = rag.search(query, top_k=5)
    print(f'\n📋 Результаты поиска: "{query}"\n')
    for i, r in enumerate(results, 1):
        title = r['meta'].get('title') or r['id']
        print(f'{i}. {title} ({r["meta"].get("type", "")})')
        print(f'   📊 Score: {r["score"]*100:.1f}% | 📁 {r["meta"].get("disk_name", "")}')
        print()

In [ ]:
# @title 💾 Скачивание индекса

from google.colab import files

print('📥 Скачайте созданный файл media_rag.db:')
files.download('media_rag.db')

print('\n📝 Инструкция:')
print('1. Скопируйте media_rag.db в папку plugins/media_organizer/data/')
print('2. Перезапустите локальное приложение')
print('3. RAG будет автоматически использован в чате')

## 📝 Использование в локальном приложении

После скачивания `media_rag.db`:

```bash
# Скопируйте файл в директорию данных
cp media_rag.db plugins/media_organizer/data/media_rag.db

# Перезапустите приложение
python main.py
```

Теперь при запросах о фильмах в чате будет использоваться RAG-поиск!